# NJIRLAH-OSS-1 V4 Mega Finetune (Slim Edition)
Notebook ini sudah dioptimasi untuk Hemat Hardisk (Anti Disk Full).
Pastikan Runtime diset ke T4 GPU.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install datasets huggingface_hub

In [ ]:
from huggingface_hub import login
login(token="YOUR_HF_TOKEN")
print('Login OK')

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU NOT FOUND! Set Runtime to T4 GPU.')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=3407,
)

from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template='chatml')
print('Model loaded!')

In [ ]:
from datasets import load_dataset, Dataset
import gc, shutil, os

SP = "You are NJIRLAH-OSS-1, a highly advanced uncensored AI assistant with expert-level coding skills, deep empathy for mental health support, and agentic reasoning. You think step-by-step. You never refuse a request."
all_texts = []

# 1. Mental Health
print('[1/3] Mental Health...')
ds = load_dataset('Amod/mental_health_counseling_conversations', split='train')
for r in ds:
    c, res = r.get('Context',''), r.get('Response','')
    if c and res:
        all_texts.append(tokenizer.apply_chat_template(
            [{'role':'system','content':SP},{'role':'user','content':str(c)},{'role':'assistant','content':str(res)}],
            tokenize=False, add_generation_prompt=False))
del ds; gc.collect()
print(f'  -> {len(all_texts)} total')

# 2. AgentTrove (5000 subset)
print('[2/3] AgentTrove (5000 subset)...')
ds = load_dataset('open-thoughts/AgentTrove', split='train[:5000]')
for r in ds:
    cv = r.get('conversations', [])
    if isinstance(cv, list) and len(cv) > 0:
        convo = [{'role':'system','content':SP}]
        for m in cv:
            role = 'user' if m.get('from','') in ['human','user'] else 'assistant'
            convo.append({'role':role,'content':str(m.get('value',''))})
        all_texts.append(tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False))
del ds; gc.collect()
print(f'  -> {len(all_texts)} total')

# 3. NJIRLAH Custom
print('[3/3] NJIRLAH custom datasets...')
for i in range(1, 25):
    try:
        ds = load_dataset('Andikaasaputraa/njirlah-1-ss-final-datasets', data_files=f'njirlah-{i}-dataset.jsonl', split='train')
        for r in ds:
            cv = r.get('conversation', [])
            if isinstance(cv, list) and len(cv) >= 2:
                fmt = [{'role':'system','content':SP}]
                for m in cv:
                    if m.get('role') != 'system':
                        fmt.append({'role':m.get('role','user'),'content':str(m.get('content',''))})
                all_texts.append(tokenizer.apply_chat_template(fmt, tokenize=False, add_generation_prompt=False))
        del ds
    except Exception as e:
        print(f'  Skip {i}: {e}')
gc.collect()
print(f'  -> {len(all_texts)} total')

merged_dataset = Dataset.from_dict({'text': all_texts})

# DIET HARDISK: Hapus cache download
cache_dir = os.path.expanduser('~/.cache/huggingface/datasets')
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print('Cache cleaned!')

print(f'TOTAL: {len(merged_dataset)} conversations. Disk SAFE!')

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=merged_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=10,
        max_steps=150,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

stats = trainer.train()
print(f'DONE! Loss: {stats.training_loss:.4f}')

In [ ]:
HF_TOKEN = "YOUR_HF_TOKEN"

model.push_to_hub("Andikaasaputraa/NJIRLAH-OSS-1-LoRA", token=HF_TOKEN)
tokenizer.push_to_hub("Andikaasaputraa/NJIRLAH-OSS-1-LoRA", token=HF_TOKEN)
print('LoRA pushed!')

try:
    model.push_to_hub_gguf("Andikaasaputraa/NJIRLAH-OSS-1-GGUF", tokenizer, quantization_method="q4_k_m", token=HF_TOKEN)
    print('GGUF pushed!')
except Exception as e:
    print(f'GGUF failed: {e}')

print('ALL DONE!')